<a href="http://landlab.github.io"><img style="float: left; width: 300px;" src="https://landlab.csdms.io/_static/landlab_logo.png"></a>

# 2D Surface Water Flow: HLLC Validation — Circular Dam Break

<hr>
<small>For more Landlab tutorials, click here: <a href="https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html">https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html</a></small>
<hr>

## Overview

This notebook demonstrates the rigorous validation of the `RiverFlowDynamics_HLLC` component using a **2D circular dam break** problem with a wet downstream. 

### Theory

A circular reservoir of depth $h_L$ sits inside a wider Cartesian domain at depth $h_R$. At $t = 0$ the dam is removed; a circular bore propagates outward and a rarefaction propagates inward, eventually reflecting at the center.

Because there is no exact 2D analytical solution for this problem, we validate the Cartesian 2D solver against a highly refined **1D radial HLLC reference solver**. The axisymmetric geometric source terms for the 1D radial shallow-water equations are:

$$\frac{\partial h}{\partial t} + \frac{\partial (hu)}{\partial r} = - \frac{hu}{r}$$
$$\frac{\partial (hu)}{\partial t} + \frac{\partial (hu^2 + \frac{1}{2}gh^2)}{\partial r} = - \frac{hu^2}{r}$$

By sampling the 2D Cartesian solution along multiple radial rays and comparing them to this 1D reference, we can rigorously test the **angular symmetry (isotropy)** of the Strang operator splitting and the mass conservation of the finite-volume solver under strong 2D gradients.

### Import the needed libraries:

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid
from landlab.components import RiverFlowDynamics_HLLC

## 1. Define Simulation Parameters

We set up the parameters for both the 2D Cartesian grid and the high-resolution 1D radial reference grid.

In [ ]:
# Domain and flow configuration
g_acc = 9.81
h_L = 2.5  # inside-the-dam depth [m]
h_R = 0.5  # outside-the-dam depth [m]
r_dam = 2.5  # initial dam radius [m]
L_dom = 20.0  # domain half-width (square -L .. +L) [m]
dx = 0.05  # 2D grid spacing [m]
t_end = 0.7  # final simulated time [s]
N_RAYS = 16  # number of radial rays for symmetry sampling

# 1D radial reference grid (finer than 2D to serve as ground truth)
dr_ref = 0.01
r_max_ref = 1.4 * L_dom  # cover sqrt(2) * L corner-to-corner

print("=" * 72)
print("  B3 — 2D circular dam break (wet/wet)")
print("=" * 72)
print(f"  h_L = {h_L} m,  h_R = {h_R} m,  r_dam = {r_dam} m")
print(f"  Domain ±{L_dom} m,  dx = {dx} m,  t_end = {t_end} s")

## 2. Generate 1D Radial Reference Solution

We first integrate a custom 1D radial HLLC solver to serve as our exact reference point for the 2D spatial sampling.

In [ ]:
def hllc_1d_radial(h_L, h_R, r_dam, r_max, dr, t_end, g=9.81, cfl=0.4):
    """Solve axisymmetric SWE on a 1D radial grid by HLLC + source term."""
    nr = int(round(r_max / dr))
    r = (np.arange(nr) + 0.5) * dr
    h = np.where(r < r_dam, h_L, h_R)
    hu = np.zeros(nr)

    def wave_speeds(hL, uL, hR, uR):
        cL = np.sqrt(g * np.maximum(hL, 0))
        cR = np.sqrt(g * np.maximum(hR, 0))
        sqL, sqR = np.sqrt(np.maximum(hL, 0)), np.sqrt(np.maximum(hR, 0))
        den = sqL + sqR
        safe = den > 0
        u_roe = np.where(safe, (sqL * uL + sqR * uR) / np.where(safe, den, 1.0), 0.0)
        c_roe = np.sqrt(g * 0.5 * (hL + hR))
        SL = np.minimum(uL - cL, u_roe - c_roe)
        SR = np.maximum(uR + cR, u_roe + c_roe)
        num = hR * uR * (uR - SR) - hL * uL * (uL - SL) + 0.5 * g * (hR**2 - hL**2)
        den_s = hR * (uR - SR) - hL * (uL - SL)
        sf = np.abs(den_s) > 1e-14
        Sstar = np.where(sf, num / np.where(sf, den_s, 1.0), 0.5 * (uL + uR))
        Sstar = np.minimum(np.maximum(Sstar, SL), SR)
        return SL, SR, Sstar

    def hllc_flux(hL, uL, hR, uR):
        SL, SR, Sstar = wave_speeds(hL, uL, hR, uR)
        F1L = hL * uL
        F2L = hL * uL * uL + 0.5 * g * hL * hL
        F1R = hR * uR
        F2R = hR * uR * uR + 0.5 * g * hR * hR
        Q1L, Q2L = hL, hL * uL
        Q1R, Q2R = hR, hR * uR
        denL = np.where(np.abs(SL - Sstar) > 1e-14, SL - Sstar, 1.0)
        denR = np.where(np.abs(SR - Sstar) > 1e-14, SR - Sstar, 1.0)
        hLs = hL * (SL - uL) / denL
        hRs = hR * (SR - uR) / denR
        Q1Ls, Q2Ls = hLs, hLs * Sstar
        Q1Rs, Q2Rs = hRs, hRs * Sstar
        F1Ls = F1L + SL * (Q1Ls - Q1L)
        F2Ls = F2L + SL * (Q2Ls - Q2L)
        F1Rs = F1R + SR * (Q1Rs - Q1R)
        F2Rs = F2R + SR * (Q2Rs - Q2R)
        F1 = np.where(
            SL >= 0, F1L, np.where(Sstar >= 0, F1Ls, np.where(SR >= 0, F1Rs, F1R))
        )
        F2 = np.where(
            SL >= 0, F2L, np.where(Sstar >= 0, F2Ls, np.where(SR >= 0, F2Rs, F2R))
        )
        return F1, F2

    t = 0.0
    while t < t_end - 1e-9:
        u = np.where(h > 1e-10, hu / h, 0.0)
        c = np.sqrt(g * np.maximum(h, 0))
        dt = cfl * dr / np.max(np.abs(u) + c)
        if t + dt > t_end:
            dt = t_end - t
        hL = np.concatenate(([h[0]], h))
        hR = np.concatenate((h, [h[-1]]))
        uL = np.concatenate(([u[0]], u))
        uR = np.concatenate((u, [u[-1]]))
        uL[0], hL[0] = -uR[0], hR[0]  # Symmetry BC at r=0
        F1, F2 = hllc_flux(hL, uL, hR, uR)
        h_new = h - dt / dr * (F1[1:] - F1[:-1])
        hu_new = hu - dt / dr * (F2[1:] - F2[:-1])
        u_old = u
        h_new = h_new - dt * h * u_old / r
        hu_new = hu_new - dt * h * u_old * u_old / r
        h, hu = h_new, hu_new
        t += dt
    return r, h, hu


print("Building 1D radial reference solution...")
r_ref, h_ref, hu_ref = hllc_1d_radial(h_L, h_R, r_dam, r_max_ref, dr_ref, t_end)
print(
    f"Done. 1D ref: nr={len(r_ref)}, h_max={h_ref.max():.4f}, h_min={h_ref.min():.4f}"
)

## 3. Configure the 2D Cartesian Model

We create the Landlab grid and map the circular initial condition onto the Cartesian nodes.

In [ ]:
# Setup Landlab Grid
ncols = int(round(2 * L_dom / dx))
nrows = ncols
grid = RasterModelGrid((nrows, ncols), xy_spacing=dx)
z = grid.add_zeros("topographic__elevation", at="node")
h_f = grid.add_zeros("surface_water__depth", at="node")
eta = grid.add_zeros("surface_water__elevation", at="node")

# Center the coordinate system
x_n = grid.x_of_node - L_dom
y_n = grid.y_of_node - L_dom
r_n = np.sqrt(x_n**2 + y_n**2)

# Initial condition
h_f[:] = np.where(r_n < r_dam, h_L, h_R)
eta[:] = h_f + z

mass0 = h_f.sum() * dx * dx

# Instantiate the solver (order=1)
rfd = RiverFlowDynamics_HLLC(
    grid, mannings_n=0.0, cfl=0.45, order=1, wall_edges=()  # Transmissive all around
)

## 4. Run the 2D Simulation

The 2D wave propagates radially. Because the final time ($t=0.7$ s) is very short, we run the integration silently to completion and check for mass conservation.

In [ ]:
t0 = time.time()
print("Running 2D Cartesian simulation (order=1)...")

while rfd.elapsed_time < t_end - 1e-9:
    rfd.run_one_step()

h_2d = grid.at_node["surface_water__depth"].reshape(nrows, ncols)
x_2d = x_n.reshape(nrows, ncols)
y_2d = y_n.reshape(nrows, ncols)
r_2d = np.sqrt(x_2d**2 + y_2d**2)

mass1 = h_2d.sum() * dx * dx
mass_err = abs(mass1 - mass0) / mass0

print(f"Simulation complete in {time.time() - t0:.2f} s.")
print(f"Steps taken: {rfd._step_n}")
print(f"Mass error:  {mass_err * 100:.4f} % (Target < 0.1 %)")

## 5. Radial Interpolation & Diagnostics

To test the isotropy (directional independence) of the Strang operator splitting, we sample the 2D result along 16 radial rays starting from the center of the domain.

In [ ]:
def sample_along_rays(h_2d_field, n_rays, r_max_sample, n_points=800):
    """Linearly interpolate h along n_rays evenly-spaced rays from origin."""
    angles = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
    r_samp = np.linspace(0, r_max_sample, n_points)
    profiles = []
    for theta in angles:
        xs = r_samp * np.cos(theta)
        ys = r_samp * np.sin(theta)
        # Bilinear interpolation (sub-grid resolution)
        col = (xs + L_dom) / dx - 0.5
        row = (ys + L_dom) / dx - 0.5
        col0 = np.clip(np.floor(col).astype(int), 0, ncols - 2)
        row0 = np.clip(np.floor(row).astype(int), 0, nrows - 2)
        fc = col - col0
        fr = row - row0
        h_interp = (
            h_2d_field[row0, col0] * (1 - fr) * (1 - fc)
            + h_2d_field[row0, col0 + 1] * (1 - fr) * fc
            + h_2d_field[row0 + 1, col0] * fr * (1 - fc)
            + h_2d_field[row0 + 1, col0 + 1] * fr * fc
        )
        profiles.append(h_interp)
    return r_samp, np.asarray(profiles)


r_samp, profiles = sample_along_rays(h_2d, N_RAYS, r_max_sample=L_dom)
h_mean = profiles.mean(axis=0)
h_min = profiles.min(axis=0)
h_max = profiles.max(axis=0)
dr_samp = r_samp[1] - r_samp[0]

# 1. Front radius spread (Bore position phase noise)
h_mid = 0.5 * (float(h_ref.max()) + h_R)
r_front_per_ray = np.zeros(N_RAYS)
for k in range(N_RAYS):
    wet = profiles[k] > h_mid
    r_front_per_ray[k] = r_samp[np.where(wet)[0][-1]] if wet.any() else np.nan

r_front_mean = float(np.nanmean(r_front_per_ray))
front_radius_spread = (
    np.nanmax(r_front_per_ray) - np.nanmin(r_front_per_ray)
) / r_front_mean

# 2. Interior RMS spread (Smooth flow region inside the bore)
interior_mask = r_samp < max(0.0, r_front_mean - 2.0 * dx)
h_dev = (profiles[:, interior_mask] - h_mean[interior_mask][None, :]) / np.where(
    h_mean[interior_mask] > 1e-6, h_mean[interior_mask], 1.0
)
interior_rms_spread = float(np.sqrt(np.mean(h_dev**2)))

# 3. L1 Mean vs 1D Reference
h_ref_interp = np.interp(r_samp, r_ref, h_ref)
L1_rel = float(np.sum(np.abs(h_mean - h_ref_interp)) * dr_samp) / float(
    np.sum(np.abs(h_ref_interp)) * dr_samp
)

print(f"Front-radius spread: {front_radius_spread * 100:6.3f} %  (Target < 2 %)")
print(f"Interior RMS spread: {interior_rms_spread * 100:6.3f} %  (Target < 2 %)")
print(f"Mean radial profile L1 vs 1D ref: {L1_rel * 100:6.3f} %  (Target < 5 %)")

## 6. Visualizing the Validation

We plot the 2D spatial distribution alongside the collapsed radial profiles to visualize the bore front and structural symmetry.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (Left Panel) 2D Depth map with concentric circles overlay
ax = axes[0]
im = ax.pcolormesh(x_2d, y_2d, h_2d, shading="auto", cmap="viridis")
ax.set_title(f"Depth h at t = {rfd.elapsed_time:.3f} s  (order=1)")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label="h [m]")
for rad in [r_dam, 2 * r_dam, 3 * r_dam]:
    if rad < L_dom:
        theta_ = np.linspace(0, 2 * np.pi, 100)
        ax.plot(rad * np.cos(theta_), rad * np.sin(theta_), "w--", lw=0.5, alpha=0.5)

# (Right Panel) Radial profiles vs Reference
ax = axes[1]
for prof in profiles:
    ax.plot(r_samp, prof, color="gray", lw=0.3, alpha=0.5)

ax.fill_between(
    r_samp, h_min, h_max, color="C0", alpha=0.2, label="Ray min/max (Spread)"
)
ax.plot(r_samp, h_mean, "C0-", lw=1.5, label="Mean 2D Profile")
ax.plot(r_ref, h_ref, "k--", lw=1.2, label="1D Radial Reference")
ax.axvline(r_dam, color="brown", ls=":", lw=1.0, label="Initial dam radius")

ax.set_xlabel("Radius r [m]")
ax.set_ylabel("Depth h [m]")
ax.set_title(f"Radial Validation (L1 Error = {L1_rel * 100:.2f}%)")
ax.legend(loc="upper right", fontsize=9)
ax.set_xlim(0, 10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Interpretation of Results

This benchmark validates the accuracy and isotropy of the HLLC Riemann solver coupled with Strang splitting. Key observations:

1. **Isotropy (Symmetry):** The front-radius spread evaluates whether the bore remains perfectly circular as it crosses the Cartesian grid. Keeping this below **2%** indicates that the 1D operator splitting does not introduce severe directional bias along the diagonals.
2. **Internal Smoothness:** The interior RMS spread measures the stability of the rarefaction wave propagating inward. The low spread value means that the grid does not artificially roughen the free surface.
3. **Radial Conservation:** The low L1 relative error against the highly resolved 1D radial reference confirms that the Cartesian flux integration exactly balances the effective radial geometry over time.

-- --
### And that's it!

You have successfully validated the 2D spatial accuracy of the `RiverFlowDynamics_HLLC` component.

-- --

### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>